# Review Sentiment & Business Intelligence Engine

---

## What this notebook does

This is a lightweight NLP pipeline built entirely on standard Python libraries — no transformers, no pretrained models. It ingests product reviews (real CSV or synthetic), scores them for sentiment, classifies them into business categories, and outputs charts and a markdown report.

### Pipeline overview

```
Load reviews
     |
     v
Feature engineering  <-- keyword-based vector classification
     |
     v
Sentiment analysis   <-- lexicon scoring + tanh normalization
     |
     v
Insight extraction   <-- rule-based recommendations per vector
     |
     v
Visualizations + Report
```

### Key design decisions
- **No ML model.** Sentiment is computed with a hand-crafted signed lexicon. This is fast, interpretable, and requires zero training data.
- **Bilingual lexicon.** The reviews are Filipino-English code-switched (Taglish). Both languages are covered in the token dictionaries.
- **tanh normalization.** Raw lexicon sums are squashed to `(-1, 1)` using `tanh`, so a long review doesn't get an artificially extreme score just from word count.
- **Multi-label vector assignment.** A single review can belong to more than one business vector (e.g., it can be about both `product` and `value`).

## 1. Imports and logging setup

Only standard data science libraries are used: `pandas` for tabular data, `numpy` for math, `matplotlib` for charts. Logging is configured so every pipeline stage prints a timestamped status line — useful for debugging during a hackathon demo.

In [ ]:
import logging
import os
import re

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

print("Imports loaded.")

## 2. Configuration

All tunable parameters are centralized here. This is good engineering practice — you change one value here instead of hunting through the codebase.

| Parameter | Purpose |
|---|---|
| `POSITIVE_THRESHOLD` | Minimum tanh score to call a review "positive" |
| `NEGATIVE_THRESHOLD` | Maximum tanh score to call a review "negative" |
| `VECTORS` | The five business dimensions being tracked |

The threshold range `[-0.15, 0.15]` is intentionally narrow — reviews with weak or mixed signals fall into `neutral` rather than being forced into a binary label.

In [ ]:
DATA_PATH = "reviews.csv"
OUTPUT_DIR = "outputs"

POSITIVE_THRESHOLD = 0.15
NEGATIVE_THRESHOLD = -0.15

VECTORS: list[str] = [
    "product",
    "packaging",
    "communication",
    "retail_execution",
    "value",
]

## 3. Keyword dictionaries for vector classification

Each business vector has a list of trigger keywords. When a review contains any of these words, it gets tagged with that vector.

**Why keyword matching instead of ML classification?**
- Zero training data needed
- Fully explainable — you can always trace *why* a review was assigned to a vector
- Fast to iterate: adding a keyword is a one-line change

The tradeoff: it won't catch paraphrases (e.g., "arrived smashed" won't hit `packaging` unless "smashed" is in the list). For a hackathon scope this is acceptable.

In [ ]:
VECTOR_KEYWORDS: dict[str, list[str]] = {
    "product": [
        "performance", "formulation", "consistency", "quality",
        "thick", "tubig", "fabcon", "formula", "scent", "effect",
    ],
    "packaging": [
        "packaging", "damaged", "box", "wrap", "leaking",
        "seal", "bukas", "basag", "broken",
    ],
    "communication": [
        "communication", "seller", "message", "response",
        "reply", "nagre-respond", "nag-reply", "contact",
    ],
    "retail_execution": [
        "delivery", "legit", "courier", "arrived", "late",
        "shipping", "dispatched", "tracking", "delayed",
    ],
    "value": [
        "value", "money", "price", "pesos", "halaga",
        "sulit", "worth", "mahal", "mura", "bayad",
    ],
}

## 4. Sentiment lexicon, intensifiers, and negators

This is the core of the NLP engine. Three separate data structures work together:

### 4a. Signed lexicon
Each token maps to a float in roughly `[-1.0, 1.0]`. Positive tokens have positive weights, negative tokens have negative weights. The magnitudes are hand-calibrated — `peke` (counterfeit) gets `-1.0` because it's the worst thing you can say about a product in this context.

### 4b. Intensifiers
Words like `very`, `sobra`, `grabe` — they don't carry sentiment on their own, but they multiply the weight of the *next* sentiment token by 1.5x.

> Example: `"very good"` → `0.5 * 1.5 = 0.75`

### 4c. Negators
Words like `not`, `hindi`, `wala` flip the sign of the next 3 scored tokens. The window of 3 accounts for typical sentence structures where the negated phrase spans multiple words.

> Example: `"hindi maganda"` → `maganda` would normally score `+0.6`, but the negator flips it to `-0.6`

In [ ]:
SENTIMENT_LEXICON: dict[str, float] = {
    # positive tokens
    "great":        0.6,
    "good":         0.5,
    "excellent":    0.8,
    "best":         0.7,
    "love":         0.7,
    "awesome":      0.7,
    "satisfied":    0.6,
    "recommend":    0.5,
    "legit":        0.7,
    "authentic":    0.6,
    "fast":         0.4,
    "quick":        0.4,
    "sulit":        0.7,   # good value
    "maganda":      0.6,   # good / beautiful
    "ganda":        0.5,
    "salamat":      0.3,   # thank you (positive signal)
    "slamat":       0.3,
    "okay":         0.2,
    # negative tokens
    "bad":         -0.6,
    "poor":        -0.6,
    "terrible":    -0.8,
    "awful":       -0.8,
    "worst":       -0.9,
    "damaged":     -0.7,
    "broken":      -0.7,
    "leaking":     -0.7,
    "late":        -0.5,
    "delayed":     -0.5,
    "fake":        -0.8,
    "counterfeit": -0.8,
    "expensive":   -0.5,
    "overpriced":  -0.6,
    "disappointed":-0.7,
    "unresponsive":-0.6,
    "tubig":       -0.7,   # watered-down product
    "mahal":       -0.4,   # expensive
    "pangit":      -0.8,   # bad / ugly
    "wala":        -0.4,   # absent / none
    "mali":        -0.6,   # wrong
    "sirang":      -0.8,   # broken / ruined
    "peke":        -1.0,   # counterfeit
}

INTENSIFIERS: set[str] = {"very", "sobra", "grabe", "super", "talagang", "really", "extremely"}
NEGATORS: set[str]     = {"not", "hindi", "wala", "never", "no", "di", "hinde"}

## 5. Data loading

The loader checks for a `reviews.csv` in the working directory. If it doesn't exist, it falls back to a hardcoded synthetic dataset. This is a practical pattern for demos and testing — the pipeline always runs, regardless of whether real data is present.

**Required CSV columns:** `review_text`, `product`, `brand`, `company`

The synthetic dataset is intentionally designed to cover all vectors and include Taglish reviews, so it exercises the full pipeline.

In [ ]:
def _synthetic_data() -> pd.DataFrame:
    rows = {
        "review_text": [
            "Tubig na po yung fabcon. Hindi na tulad ng dati na thick yung consistency.",
            "Legit seller. Salamat po, packaging okay naman.",
            "Great value for money, would buy again.",
            "2 liters nabibili ko lang sa halagang 111 pesos. Sulit!",
            "Packaging arrived damaged and leaking. Box was not sealed properly.",
            "Communication from seller was excellent and fast.",
            "Delivery was late. Courier was unresponsive to tracking queries.",
            "Product performance is top-notch. Formula is very consistent.",
            "Hindi sulit. Mahal na nga, pangit pa quality ng product.",
            "Seller replied agad. Legit seller, scent is strong and long-lasting.",
            "Maganda ang quality ng product. Will definitely re-order.",
            "Poor value for price paid. Expected better formulation.",
        ],
        "product": ["Fabcon"] * 6 + ["Shampoo"] * 6,
        "brand":   ["Downy"] * 6 + ["Pantene"] * 6,
        "company": ["P&G"] * 6 + ["Unilever"] * 6,
    }
    return pd.DataFrame(rows)


def load_reviews() -> pd.DataFrame:
    if os.path.exists(DATA_PATH):
        df = pd.read_csv(DATA_PATH)
        required = {"review_text", "product", "brand", "company"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"CSV is missing required columns: {missing}")
        log.info("Loaded %d rows from %s", len(df), DATA_PATH)
    else:
        df = _synthetic_data()
        log.info("No CSV found at '%s'. Using %d-row synthetic dataset.", DATA_PATH, len(df))
    return df.dropna(subset=["review_text"]).reset_index(drop=True)

## 6. Feature engineering

`classify_vectors` is the multi-label classifier. It lowercases the review text and checks for keyword membership. It returns a list — a single review can belong to `["product", "value"]` simultaneously.

`engineer_features` builds two derived columns:
- `vectors` — the full list of matched vectors
- `primary_vector` — just the first matched vector, for cases where a single label is needed
- `cbp` — a `company / brand / product` identifier string for grouping in reports

In [ ]:
def classify_vectors(text: str) -> list[str]:
    """
    Return the list of business vectors the review touches.
    Falls back to ['other'] if no keywords match.
    """
    text_lower = text.lower()
    matched = [
        vector
        for vector, keywords in VECTOR_KEYWORDS.items()
        if any(kw in text_lower for kw in keywords)
    ]
    return matched if matched else ["other"]


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["vectors"] = df["review_text"].apply(classify_vectors)
    df["primary_vector"] = df["vectors"].apply(lambda v: v[0])
    df["cbp"] = df.apply(
        lambda r: f"{r['company']} / {r['brand']} / {r['product']}", axis=1
    )
    log.info("Feature engineering complete.")
    return df

## 7. Sentiment analysis

This is the most technically interesting part of the pipeline.

### How `_score_text` works step by step

1. Strip punctuation with regex, lowercase, split into tokens
2. Iterate through each token:
   - If it's a **negator**: set `negation_window = 3`, skip to next token
   - If it's an **intensifier**: set `prev_was_intensifier = True`, skip to next token
   - If it's in the **lexicon**: retrieve its weight, apply 1.5x if intensifier preceded it, flip sign if inside a negation window, add to running total
   - Decrement `negation_window` on every token (scored or not)
3. Apply `tanh` to the total

### Why tanh?
`tanh(x)` maps any real number to `(-1, 1)`. Without it, a review with 10 positive words would score `~5.0` while a review with 2 positive words scores `~1.0` — length dominates. With tanh, both saturate near `+1` if they're strongly positive, which is the correct behavior.

```
tanh(0.5)  ≈  0.46   (weak positive)
tanh(1.0)  ≈  0.76   (moderate positive)
tanh(2.0)  ≈  0.96   (strong positive)
tanh(5.0)  ≈  0.9999 (saturated)
```

In [ ]:
def _score_text(text: str) -> float:
    """
    Lexicon-based sentiment score with intensifier and negation handling.
    Returns a value in (-1, 1) via tanh normalization.
    """
    tokens = re.sub(r"[^\w\s]", "", text.lower()).split()

    total = 0.0
    prev_was_intensifier = False
    negation_window = 0  # counts down from 3 after a negator is hit

    for token in tokens:
        if token in NEGATORS:
            negation_window = 3
            prev_was_intensifier = False
            continue

        if token in INTENSIFIERS:
            prev_was_intensifier = True
            continue

        weight = SENTIMENT_LEXICON.get(token, 0.0)
        if weight != 0.0:
            if prev_was_intensifier:
                weight *= 1.5
            if negation_window > 0:
                weight *= -1
                negation_window -= 1
            total += weight

        prev_was_intensifier = False
        if negation_window > 0:
            negation_window -= 1

    return float(np.tanh(total))


def analyze_sentiment(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["sentiment_score"] = df["review_text"].apply(_score_text)
    df["sentiment_label"] = df["sentiment_score"].apply(
        lambda s: "positive"
        if s > POSITIVE_THRESHOLD
        else ("negative" if s < NEGATIVE_THRESHOLD else "neutral")
    )
    dist = df["sentiment_label"].value_counts().to_dict()
    log.info("Sentiment analysis complete. Distribution: %s", dist)
    return df

### Quick test: score individual reviews

Run this cell to verify the scorer is behaving correctly before running the full pipeline.

In [ ]:
test_cases = [
    "Legit seller, very fast delivery, sulit!",
    "Packaging was damaged and leaking. Box was not sealed.",
    "Hindi sulit. Mahal na nga, pangit pa.",
    "Okay naman.",  # Expected: near-neutral
    "Not good at all.",  # Negation should flip 'good'
]

for t in test_cases:
    score = _score_text(t)
    label = "positive" if score > POSITIVE_THRESHOLD else ("negative" if score < NEGATIVE_THRESHOLD else "neutral")
    print(f"[{label:>8}  {score:+.3f}]  {t}")

## 8. Insight extraction

For each business vector, the engine computes:
- Total mention count
- Average sentiment score
- Brands with the most negative mentions

It then looks up a pre-written recommendation from `_RECOMMENDATIONS` based on whether the vector's overall signal is positive or negative. This is a **rule-based decision layer** — think of it as if-else logic wrapped in a dictionary.

This approach is appropriate here because the action space is well-defined: you either have a negative packaging problem or you don't. More complex scenarios would warrant a generative output layer.

In [ ]:
_RECOMMENDATIONS: dict[str, dict[str, str]] = {
    "product": {
        "negative": (
            "Launch a quality-reassurance campaign. Highlight formula "
            "consistency and product authenticity in listings and content."
        ),
        "positive": (
            "Leverage positive product sentiment in paid media. "
            "Feature top reviews in product detail pages."
        ),
    },
    "packaging": {
        "negative": (
            "Audit packaging supplier for seal and structural integrity. "
            "Add a replacement guarantee to listings to reduce churn from damaged orders."
        ),
        "positive": (
            "Packaging is a differentiator. Maintain standards and "
            "highlight unboxing experience in social content."
        ),
    },
    "value": {
        "negative": (
            "Introduce a bundle promo or trial SKU to address price sensitivity. "
            "Communicate cost-per-use versus competitors."
        ),
        "positive": (
            "Value perception is strong. Reinforce with best-value badging "
            "and loyalty program incentives."
        ),
    },
    "retail_execution": {
        "negative": (
            "Escalate courier SLA breaches. Conduct fulfillment partner audit "
            "and surface real-time tracking in post-purchase communications."
        ),
        "positive": (
            "Delivery execution is working. Benchmark current courier partners "
            "and replicate model across other SKUs."
        ),
    },
    "communication": {
        "negative": (
            "Implement auto-reply templates for seller response time. "
            "Target sub-1-hour first response on marketplace channels."
        ),
        "positive": (
            "Seller communication is a competitive advantage. "
            "Document response playbooks and scale to new agents."
        ),
    },
}


def extract_insights(df: pd.DataFrame) -> dict[str, list[str]]:
    # explode() turns each row with a list of vectors into multiple rows,
    # one per vector — standard technique for multi-label analysis in pandas
    exploded = df.explode("vectors").rename(columns={"vectors": "vector"})
    insights: dict[str, list[str]] = {}

    for vector in VECTORS:
        vec_df = exploded[exploded["vector"] == vector]
        if vec_df.empty:
            continue

        neg_df = vec_df[vec_df["sentiment_label"] == "negative"]
        pos_df = vec_df[vec_df["sentiment_label"] == "positive"]
        total_mentions = len(vec_df)
        avg_score = vec_df["sentiment_score"].mean()

        lines: list[str] = [
            f"Mentions: {total_mentions} | "
            f"Avg sentiment score: {avg_score:.3f} | "
            f"Negative: {len(neg_df)} | Positive: {len(pos_df)}"
        ]

        if not neg_df.empty:
            affected = neg_df["brand"].value_counts().head(3).to_dict()
            lines.append(f"Brands with negative mentions: {affected}")
            rec = _RECOMMENDATIONS.get(vector, {}).get(
                "negative", "Review negative feedback manually."
            )
            lines.append(f"Recommended action: {rec}")
        else:
            top = pos_df["brand"].value_counts().head(3).to_dict() if not pos_df.empty else {}
            lines.append(f"No negative mentions. Leading brands: {top}")
            rec = _RECOMMENDATIONS.get(vector, {}).get(
                "positive", "Maintain current performance."
            )
            lines.append(f"Recommended action: {rec}")

        insights[vector] = lines

    log.info("Insight extraction complete for %d vectors.", len(insights))
    return insights

## 9. Visualizations

Three charts are generated and saved to the `outputs/` directory.

| Chart | What it shows |
|---|---|
| `sentiment_by_vector.png` | Bar chart: average sentiment score per vector |
| `heatmap_brand_vector.png` | Heatmap: brand × vector sentiment matrix |
| `sentiment_distribution.png` | Pie chart: positive / neutral / negative split |

**Implementation note on the heatmap:** `df.explode("vectors")` + `groupby(["brand", "vector"])` + `.unstack()` is the standard pandas idiom for converting long-format multi-label data into a pivot table. `imshow` from matplotlib then renders it as a color grid.

In [ ]:
def _bar_chart(vec_avg: pd.Series) -> None:
    colors = ["#c0392b" if v < 0 else "#27ae60" for v in vec_avg.values]
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(
        vec_avg.index, vec_avg.values,
        color=colors, edgecolor="white", linewidth=0.6,
    )
    ax.axhline(0, color="#7f8c8d", linewidth=0.8, linestyle="--")
    ax.set_title("Average Sentiment Score per Business Vector", fontsize=13, fontweight="bold")
    ax.set_xlabel("Vector")
    ax.set_ylabel("Sentiment Score (tanh-normalised)")
    ax.set_ylim(-1.1, 1.1)
    for bar, val in zip(bars, vec_avg.values):
        offset = 0.03 if val >= 0 else -0.07
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            val + offset,
            f"{val:.3f}",
            ha="center", va="bottom", fontsize=9,
        )
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "sentiment_by_vector.png"), dpi=150)
    plt.show()
    plt.close()
    log.info("Saved sentiment_by_vector.png")


def _heatmap(pivot: pd.DataFrame) -> None:
    if pivot.empty:
        log.warning("Heatmap skipped: pivot table is empty.")
        return
    cmap = mcolors.LinearSegmentedColormap.from_list(
        "sentiment", ["#c0392b", "#ecf0f1", "#27ae60"]
    )
    fig, ax = plt.subplots(figsize=(10, max(3, len(pivot) * 0.9)))
    im = ax.imshow(pivot.values, cmap=cmap, vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=30, ha="right", fontsize=10)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=10)
    plt.colorbar(im, ax=ax, label="Avg Sentiment Score")
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center", fontsize=9)
    ax.set_title("Brand x Vector Sentiment Heatmap", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "heatmap_brand_vector.png"), dpi=150)
    plt.show()
    plt.close()
    log.info("Saved heatmap_brand_vector.png")


def _pie_chart(label_counts: pd.Series) -> None:
    palette = {"positive": "#27ae60", "neutral": "#f39c12", "negative": "#c0392b"}
    colors = [palette.get(lbl, "#95a5a6") for lbl in label_counts.index]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(
        label_counts.values,
        labels=label_counts.index,
        colors=colors,
        autopct="%1.1f%%",
        startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 1.2},
    )
    ax.set_title("Overall Sentiment Distribution", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "sentiment_distribution.png"), dpi=150)
    plt.show()
    plt.close()
    log.info("Saved sentiment_distribution.png")


def generate_visualizations(df: pd.DataFrame) -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    exploded = df.explode("vectors").rename(columns={"vectors": "vector"})
    active_vectors = [v for v in VECTORS if v in exploded["vector"].values]

    vec_avg = (
        exploded[exploded["vector"].isin(active_vectors)]
        .groupby("vector")["sentiment_score"]
        .mean()
        .reindex(active_vectors)
        .fillna(0)
    )
    _bar_chart(vec_avg)

    pivot = (
        exploded[exploded["vector"].isin(active_vectors)]
        .groupby(["brand", "vector"])["sentiment_score"]
        .mean()
        .unstack(fill_value=0)
        .reindex(columns=active_vectors, fill_value=0)
    )
    _heatmap(pivot)
    _pie_chart(df["sentiment_label"].value_counts())

## 10. Report generation

The report is written as a markdown file to `outputs/report.md`. It contains:
- Executive summary table (counts and percentages per sentiment label)
- Methodology note explaining the scoring approach
- Per-vector insights and recommended actions
- Brand × vector summary as a markdown table
- Chart index

This is a straightforward file-write with f-string templating. In a production system you'd likely use a templating engine like Jinja2, but for a hackathon scope this is clean and sufficient.

In [ ]:
def generate_report(df: pd.DataFrame, insights: dict[str, list[str]]) -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    path = os.path.join(OUTPUT_DIR, "report.md")

    total = len(df)
    pos   = (df["sentiment_label"] == "positive").sum()
    neg   = (df["sentiment_label"] == "negative").sum()
    neu   = (df["sentiment_label"] == "neutral").sum()
    avg   = df["sentiment_score"].mean()

    with open(path, "w", encoding="utf-8") as f:
        f.write("# Business Intelligence Report — Review Sentiment Analysis\n\n")

        f.write("## Executive Summary\n\n")
        f.write("| Metric | Value |\n|---|---|\n")
        f.write(f"| Total reviews | {total} |\n")
        f.write(f"| Positive | {pos} ({pos / total * 100:.1f}%) |\n")
        f.write(f"| Neutral | {neu} ({neu / total * 100:.1f}%) |\n")
        f.write(f"| Negative | {neg} ({neg / total * 100:.1f}%) |\n")
        f.write(f"| Overall avg sentiment score | {avg:.3f} |\n\n")

        f.write("## Methodology\n\n")
        f.write(
            "Sentiment is scored using a signed lexicon of English and Filipino/Tagalog tokens, "
            "with intensifier multiplication and negation-window flipping. "
            "Raw sums are normalised to [-1, 1] via tanh so that longer reviews "
            "do not artificially dominate the scale.\n\n"
            "Reviews are assigned to one or more business vectors via keyword matching (multi-label). "
            "Insight rules are applied per vector using negative mention volume "
            "and average score as the primary signals.\n\n"
        )

        f.write("## Vector Insights and Recommended Actions\n\n")
        for vector in VECTORS:
            if vector not in insights:
                continue
            f.write(f"### {vector.replace('_', ' ').title()}\n\n")
            for line in insights[vector]:
                f.write(f"{line}\n\n")

        f.write("## Brand x Vector Summary\n\n")
        exploded = df.explode("vectors").rename(columns={"vectors": "vector"})
        summary = (
            exploded[exploded["vector"].isin(VECTORS)]
            .groupby(["brand", "vector"])["sentiment_score"]
            .agg(avg_score="mean", mention_count="count")
            .round(3)
            .reset_index()
        )
        headers   = "| " + " | ".join(summary.columns) + " |"
        separator = "| " + " | ".join(["---"] * len(summary.columns)) + " |"
        rows = "\n".join(
            "| " + " | ".join(str(v) for v in row) + " |"
            for row in summary.itertuples(index=False)
        )
        f.write(f"{headers}\n{separator}\n{rows}")
        f.write("\n\n")

        f.write("## Charts\n\n")
        f.write("- `sentiment_by_vector.png` — average sentiment score per business vector\n")
        f.write("- `heatmap_brand_vector.png` — brand x vector sentiment heatmap\n")
        f.write("- `sentiment_distribution.png` — overall positive / neutral / negative split\n")

    log.info("Report saved to %s", path)

## 11. Run the full pipeline

Execute this cell to run all stages end-to-end. Outputs will be written to `outputs/`.

If you have a `reviews.csv`, place it in the same directory as this notebook before running.

In [ ]:
df = load_reviews()
df = engineer_features(df)
df = analyze_sentiment(df)
insights = extract_insights(df)
generate_visualizations(df)
generate_report(df, insights)

log.info("Pipeline complete. Outputs in '%s/'.", OUTPUT_DIR)

## 12. Inspect the output DataFrame

After running the pipeline, inspect the processed data to verify everything is correct before looking at the charts.

In [ ]:
# Preview the scored reviews
df[["review_text", "vectors", "primary_vector", "sentiment_score", "sentiment_label"]].head(12)

In [ ]:
# Sentiment distribution
print(df["sentiment_label"].value_counts())
print(f"\nMean score: {df['sentiment_score'].mean():.4f}")

In [ ]:
# Per-vector insights summary
for vector, lines in insights.items():
    print(f"\n--- {vector.upper()} ---")
    for line in lines:
        print(line)